# Penalty cap — optimal bid and delivery rate (pre-planned revert)

A model of how a solver responds to the **penalty cap** $c_l$. The solver **may walk away** from a
bad routing draw and pay the penalty instead of delivering — but it has to decide on that revert
rule **before the reference score is revealed**, so the rule cannot depend on the realised
$s_{\text{ref}}$, only on its win-conditional expectation.

A solver bids $s$ and wins the auction iff $s \ge s_{\text{ref}}$ (the competitor's reference bid).
On a win it draws a routing surplus $\sigma$ from a distribution ($\sigma = 0$ means the routing
failed). Conditional on winning:

- **reward** (paid on delivery): $R = \min(c_u,\, s - s_{\text{ref}})$
- **penalty** (paid on default): $P = \min(c_l,\, s_{\text{ref}})$
- **deliver payoff** $R + \sigma - s$,  **default payoff** $-P$.

The solver pre-plans a **single revert threshold** $\bar\sigma(s)$ and delivers only when the
realised surplus clears it:

$$\text{deliver} \iff \sigma \ge \bar\sigma(s), \qquad
  \bar\sigma(s) = \mathbb{E}\big[\sigma^*(s, s_{\text{ref}}) \mid \text{win}\big], \qquad
  \sigma^*(s, s_{\text{ref}}) = s - R - P.$$

$\bar\sigma(s)$ is the win-conditional **mean** of the per-realisation break-even threshold, so the
walk-away decision uses $\sigma$ and the *expected* reference score but never the realised
$s_{\text{ref}}$. (If $\sigma = 0$ the routing failed and default is forced regardless.)

Primitives follow the bidding model in
[`../bidding/bidding.ipynb`](../../bidding/bidding.ipynb) (the *selective* / pre-planned-revert
regime); in this repo's vocabulary $c_l \leftrightarrow$ `penalty_cap` and
$c_u \leftrightarrow$ `reward_cap_upper`.

**Question.** For each penalty cap $c_l$ the solver re-optimises its bid
$s^*(c_l) = \arg\max_s \Pi(s)$. We sweep $c_l$ along the x-axis and track:

- **left panel:** the optimal bid $s^*(c_l)$,
- **right panel:** the **delivery rate** at that bid — $P(\text{deliver} \mid \text{win}) = 1 - \text{revert rate}$.

Sliders control the routing ($\mu, \sigma_{\text{std}}, p_{\text{fail}}, n_{\text{bins}}$) and the
reward cap $c_u$ (held fixed while $c_l$ sweeps).

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

In [ ]:
# ----------------------------------------------------------------------
# Model (pre-planned revert).  A `routing` is a pair of arrays (sig, p);
# sig = 0 marks a routing failure (forced default).  The solver delivers
# iff the realised surplus clears a single threshold sigma_bar(s) fixed
# before s_ref is revealed: deliver iff sigma >= sigma_bar(s).
# ----------------------------------------------------------------------
def normalize(sig, p):
    sig = np.asarray(sig, float); p = np.asarray(p, float)
    return sig, p / p.sum()

def with_failure(sig, p, p_fail):
    """Mix a positive-surplus routing with mass `p_fail` at sigma = 0."""
    sig = np.asarray(sig, float); p = np.asarray(p, float)
    return normalize(np.concatenate([[0.0], sig]),
                     np.concatenate([[p_fail], (1 - p_fail) * p]))

def discretize_density(density, lo, hi, n):
    edges = np.linspace(lo, hi, n + 1)
    mids = 0.5 * (edges[:-1] + edges[1:])
    w = np.array([density(m) for m in mids], float)
    return normalize(mids, w)

def gaussian_routing(mu, sigstd, lo, hi, n):
    return discretize_density(lambda x: np.exp(-0.5 * ((x - mu) / sigstd) ** 2), lo, hi, n)


def _partial_moments(sig, p):
    """Build q_and_mu(tau) for arrays tau:
        q(tau)  = P(sigma > 0 and sigma >= tau)
        mu(tau) = E[sigma * 1{sigma > 0 and sigma >= tau}]
    via suffix sums over the positive-surplus atoms."""
    pos = sig > 0
    s = sig[pos]; pp = p[pos]
    order = np.argsort(s)
    s_sorted = s[order]; pp_sorted = pp[order]
    suf_p = np.concatenate([np.cumsum(pp_sorted[::-1])[::-1], [0.0]])
    suf_mu = np.concatenate([np.cumsum((pp_sorted * s_sorted)[::-1])[::-1], [0.0]])

    def q_and_mu(tau):
        idx = np.searchsorted(s_sorted, tau, side="left")  # count of atoms < tau
        return suf_p[idx], suf_mu[idx]

    return q_and_mu


def model_curves(routing, S, S_ref, c_u, c_l):
    """Expected payoff Pi(s) over the bid grid S and the delivery rate
    (conditional on winning) over S, under a pre-planned revert threshold.

    Pi(s) = (1/|S_ref|) * sum_{s_ref} 1{win} * E_sigma[payoff]   (losing -> 0).
    The threshold sigma_bar(s) = mean over winning s_ref of sigma*(s, s_ref)
    is one number per bid (it never sees the realised s_ref), so the deliver
    probability q(sigma_bar) and surplus mu(sigma_bar) factor out per s_ref:

        payoff = q (R - s) + mu  -  (1 - q) P,   with q = q(sigma_bar(s))."""
    sig, p = routing
    q_and_mu = _partial_moments(sig, p)

    Sm = S[:, None]; Sr = S_ref[None, :]
    win = Sm >= Sr                                  # (ns, nr)
    R = np.minimum(c_u, Sm - Sr)                    # reward, valid where win
    P = np.minimum(c_l, S_ref)[None, :]             # penalty, by s_ref
    nwin = win.sum(1); nwin_safe = np.maximum(nwin, 1)

    # sigma_bar(s) = E[sigma*(s, s_ref) | win], one threshold per bid s.
    sigstar = Sm - R - P                            # (ns, nr)
    sbar = np.where(win, sigstar, 0.0).sum(1) / nwin_safe
    sbar = np.where(nwin > 0, sbar, 0.0)            # (ns,)

    q, mu = q_and_mu(sbar)                          # (ns,) deliver prob & surplus above threshold
    g = q[:, None] * R + mu[:, None] - q[:, None] * Sm - (1 - q)[:, None] * P
    Pi = np.where(win, g, 0.0).sum(1) / len(S_ref)
    deliv = q                                       # P(deliver | win) = q(sigma_bar(s))
    return dict(Pi=Pi, deliv=deliv, win_prob=nwin / len(S_ref))


def optima_for_cap(routing, S, S_ref, c_u, c_l):
    """argmax bid s* and the delivery rate / win prob at s*."""
    cur = model_curves(routing, S, S_ref, c_u, c_l)
    i = int(np.argmax(cur["Pi"]))
    return dict(s_star=float(S[i]), deliv=float(cur["deliv"][i]), win=float(cur["win_prob"][i]))


def cap_sweep(routing, S, S_ref, c_u, c_l_grid):
    """Optimal bid, delivery rate and win prob for each penalty cap in the grid."""
    keys = ("s_star", "deliv", "win")
    res = {k: [] for k in keys}
    for c_l in c_l_grid:
        o = optima_for_cap(routing, S, S_ref, c_u, c_l)
        for k in keys:
            res[k].append(o[k])
    return {k: np.asarray(v) for k, v in res.items()}

In [ ]:
# Sanity checks: the delivery rate stays at or below the ceiling q = 1 - p_fail,
# dips when the penalty is cheap (the solver walks away from poor draws) and
# rises monotonically back to the ceiling as c_l grows.
def _selfcheck():
    S = np.arange(0, 201, 1.0); S_ref = np.arange(0, 201, 2.0)
    r = with_failure(*gaussian_routing(120, 30, 0, 200, 24), 0.15)
    q = float(r[1][r[0] > 0].sum())
    sw = cap_sweep(r, S, S_ref, 200.0, np.linspace(0, 250, 51))
    d = sw["deliv"]
    assert (d >= -1e-9).all() and (d <= q + 1e-9).all(), "delivery outside [0, q]"
    assert (np.diff(d) >= -1e-9).all(), "delivery not monotone in c_l"
    assert abs(d[-1] - q) < 1e-6, "delivery does not converge to ceiling q"
    print(f"self-check OK  (ceiling q = 1 - p_fail = {q:.3f}; "
          f"delivery dips to {d[0]:.3f} at c_l = 0, recovers to {d[-1]:.3f})")

_selfcheck()

In [ ]:
# ----------------------------------------------------------------------
# Interactive figure: optimal bid s*(c_l) and delivery rate at s* as the
# penalty cap c_l sweeps the x-axis.  Sliders set the routing and the
# (fixed) reward cap c_u.
# ----------------------------------------------------------------------
S       = np.arange(0, 201, 1.0)     # bid grid
S_ref   = np.arange(0, 201, 2.0)     # competitor reference grid (uniform)
N_CAPS  = 51                         # resolution of the c_l sweep

plt.ioff()
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11.5, 4.6), constrained_layout=True)
plt.ion()
fig.canvas.header_visible = False

(lineL,) = axL.plot([], [], color="dodgerblue", lw=2)
(lineR,) = axR.plot([], [], color="dodgerblue", lw=2, label="delivery rate")
ceil_line = axR.axhline(0.0, ls="--", lw=1, color="gray", alpha=0.8)

axL.set(xlabel="penalty cap  $c_l$", ylabel="optimal bid  $s^*$", title="Optimal bid")
axR.set(xlabel="penalty cap  $c_l$", ylabel="delivery rate   $P(\\mathrm{deliver}\\mid\\mathrm{win})$",
        title="Delivery rate at $s^*$")

sl = dict(
    mu      = widgets.FloatSlider(value=120, min=0,    max=300, step=1,    description="μ"),
    sigstd  = widgets.FloatSlider(value=30,  min=1,    max=100, step=1,    description="σ_std"),
    p_fail  = widgets.FloatSlider(value=0.15,min=0,    max=1,   step=0.01, description="p_fail",
                                  readout_format=".2f"),
    n_bins  = widgets.IntSlider  (value=24,  min=4,    max=64,  step=1,    description="n_bins"),
    c_u     = widgets.FloatSlider(value=200, min=0,    max=500, step=5,    description="c_u (reward)"),
    c_l_max = widgets.FloatSlider(value=250, min=20,   max=500, step=10,   description="c_l max"),
)

def update(_=None):
    v = {k: w.value for k, w in sl.items()}
    routing = with_failure(*gaussian_routing(v["mu"], v["sigstd"], 0, 200, int(v["n_bins"])),
                           v["p_fail"])
    c_l_grid = np.linspace(0, v["c_l_max"], N_CAPS)
    sw = cap_sweep(routing, S, S_ref, v["c_u"], c_l_grid)

    lineL.set_data(c_l_grid, sw["s_star"])
    lineR.set_data(c_l_grid, sw["deliv"])

    q = 1 - v["p_fail"]
    ceil_line.set_ydata([q, q])
    ceil_line.set_label(f"ceiling  $1-p_{{fail}}$ = {q:.2f}")

    axL.set_xlim(0, v["c_l_max"]); axR.set_xlim(0, v["c_l_max"])
    axL.set_ylim(0, max(1.0, float(sw["s_star"].max())) * 1.08)
    # zoom the delivery axis to its actual range (the dip is often small) while
    # keeping the ceiling in view.
    dmin = float(sw["deliv"].min()); span = max(q - dmin, 0.02)
    axR.set_ylim(max(0.0, dmin - 0.2 * span), min(1.02, q + 0.2 * span))
    axR.legend(loc="best", fontsize=8)
    fig.suptitle(f"μ={v['mu']:.0f}   σ_std={v['sigstd']:.0f}   p_fail={v['p_fail']:.2f}   "
                 f"n_bins={int(v['n_bins'])}   reward cap c_u={v['c_u']:.0f}", fontsize=9)
    fig.canvas.draw_idle()

for w in sl.values():
    w.observe(update, names="value")
update()

display(widgets.HBox([widgets.VBox(list(sl.values())), fig.canvas]))

### Reading the figure

- **Delivery rate (right).** When the penalty is cheap the threshold $\bar\sigma(s)$ is high, so the
  solver pre-plans to walk away from poor routing draws and the delivery rate **dips** below the
  dashed ceiling $1 - p_{\text{fail}}$. As $c_l$ rises, defaulting gets expensive, $\bar\sigma(s)$
  falls, and the delivery rate climbs back to the ceiling — where the solver only fails when the
  routing itself fails.
- **Optimal bid (left).** A cheap walk-away option lowers the effective "must-deliver" probability,
  so at small $c_l$ an extra unit of bid is cheaper at the margin and the solver bids a little more
  aggressively to win. As $c_l$ grows the option loses value and the optimal bid settles down.

Because the revert threshold uses only the *expected* reference score, both effects are milder than
in the ex-post-default ("defer", `mode = :defer` in `bidding.ipynb`) model, where the solver reacts
to the realised $s_{\text{ref}}$ and can walk away far more opportunistically. `cap_sweep` also
returns `win` ($P(s^* \ge s_{\text{ref}})$) if you want to plot the win rate alongside.

*(The right panel's y-axis is zoomed to the delivery range — read the tick values — since the dip is
typically a fraction of a percent; raising `p_fail` makes it larger.)*